# Subject Identity Strongly Shapes EEG Spectral Features and Can Inflate Machine-Learning Performance

**Analysis notebook** — companion code by Ugail & Howard.

Reproduces all tables and figures in the manuscript across four public EEG datasets:

| Dataset | Subjects | Paradigm | Role |
|---------|----------|----------|------|
| Mumtaz MDD | 63 | Resting-state (clinical) | Leakage simulation |
| SignEEG v1.0 | 70 | Signature imagery (5-ch Emotiv) | Boundary condition |
| Longitudinal ERP | 395 | RSVP face-target | Variance decomposition, cross-session |
| PhysioNet EEGMMIDB | 103 | Motor imagery | Replication |

**Prerequisites:** Download and preprocess each dataset — see `README.md` for links and preprocessing notebooks.

**Runtime:** ~20–30 min on a standard CPU (Sections A–I run sequentially). GPU (A100 or equivalent) is highly recommended.

Preprocessed datastets are available at:
https://drive.google.com/drive/folders/1LdPIw7vNINWq_PIYkj9-S0nVJ5MzJtag?usp=sharing


## 0 · Configuration

Set `BASE_DIR` to the folder that contains your `processed/` subdirectories.  
On Google Colab, mount Drive first then set the path below.

In [ ]:
import os
from pathlib import Path

# ── Set your data root here ───────────────────────────────────────────────────
# Default: look for a 'data/' folder next to this notebook.
# On Colab:  BASE_DIR = Path("/content/drive/MyDrive/your/path")
BASE_DIR = Path(os.getenv("EEG_DATA_DIR", "data"))

PATHS = {
    "mumtaz":       BASE_DIR / "Mumtaz"            / "processed",
    "signeeg":      BASE_DIR / "signeeg"           / "processed",
    "longitudinal": BASE_DIR / "longitudinal_erp"  / "processed",
    "physionet":    BASE_DIR / "physionet_eegmmidb" / "processed",
}
OUT_DIR = BASE_DIR / "results"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Data paths:")
for k, p in PATHS.items():
    status = "OK" if p.exists() else "NOT FOUND"
    print(f"  {k:<15} [{status}]  {p}")
print(f"\nResults will be saved to: {OUT_DIR}")


## 1 · Imports

In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu, gaussian_kde
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score,
    roc_curve,
    top_k_accuracy_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler

warnings.filterwarnings("ignore")
np.random.seed(42)

plt.rcParams.update({
    "font.size": 11, "axes.titlesize": 13, "axes.labelsize": 12,
    "figure.dpi": 130, "savefig.dpi": 300,
    "axes.spines.top": False, "axes.spines.right": False,
})

# ── Experiment constants ──────────────────────────────────────────────────────
N_REPEATS         = 10
TRAIN_SEG_FRAC    = 0.70
TRAIN_SUBJ_FRAC   = 0.70
NEG_PAIRS_PER_POS = 2
MIN_SEGS          = 6

ET = dict(n_estimators=300, random_state=42, n_jobs=-1)
LR = dict(max_iter=1000, random_state=42, C=1.0, solver="lbfgs", n_jobs=-1)

print("Imports OK.")


## 2 · Load datasets

Reads each preprocessed `feature_table.csv`.  
Applies identifier fixes so subject keys are unambiguous across all datasets.

In [ ]:
def load_ds(name):
    p  = PATHS[name]
    ft = pd.read_csv(p / "feature_table.csv", low_memory=False)
    fc = [c for c in ft.columns if "__" in c]
    manifest = json.load(open(p / "manifest.json")) if (p / "manifest.json").exists() else {}
    return {"name": name, "ft": ft, "feat_cols": fc, "manifest": manifest}

print("Loading …")
DS = {n: load_ds(n) for n in PATHS}

# Fix 1: Mumtaz — prepend clinical label to avoid H / MDD ID collision
ft = DS["mumtaz"]["ft"]
if "label" in ft.columns:
    ft["subject_key"] = ft["label"].astype(str) + "_" + ft["subject_id"].astype(str)
    DS["mumtaz"]["ft"] = ft
print(f"  mumtaz      subject_key: {ft['subject_key'].nunique()} subjects")

# Fix 2: SignEEG — long numeric IDs read as float64; strip trailing '.0'
ft = DS["signeeg"]["ft"]
ft["subject_id"]  = ft["subject_id"].astype(str).str.replace(r"\.0$", "", regex=True)
ft["subject_key"] = "H_" + ft["subject_id"]
DS["signeeg"]["ft"] = ft
print(f"  signeeg     subject_key: {ft['subject_key'].nunique()} subjects "
      f"(note: identifiers may be provisional — see paper §3.1)")

# Fix 3: Longitudinal — ensure subject_key column exists
ft = DS["longitudinal"]["ft"]
if "subject_key" not in ft.columns:
    ft["subject_key"] = ft["subject_id"].astype(str)
    DS["longitudinal"]["ft"] = ft

# Fix 4: PhysioNet — prefix H_ for consistency
ft = DS["physionet"]["ft"]
if "subject_key" not in ft.columns:
    ft["subject_key"] = "H_" + ft["subject_id"].astype(str)
    DS["physionet"]["ft"] = ft

print()
for name, ds in DS.items():
    ft = ds["ft"]
    print(f"  {name:<15} {len(ft):>8,} segments   "
          f"{ft['subject_key'].nunique():>4} subjects   "
          f"{len(ds['feat_cols']):>4} features")


## 3 · Shared helpers

In [ ]:
def pipe(clf):
    return Pipeline([("imp", SimpleImputer(strategy="median")),
                     ("sc",  StandardScaler()), ("clf", clf)])


def eer(fpr, tpr):
    fnr = 1.0 - tpr
    i   = np.nanargmin(np.abs(fnr - fpr))
    return float(np.mean([fpr[i], fnr[i]]))


def seg_split(df, feat_cols, frac, seed, min_segs=MIN_SEGS):
    # Split segments 70/30 within each subject independently.
    rng    = np.random.default_rng(seed)
    counts = df.groupby("subject_key").size()
    valid  = counts[counts >= min_segs].index
    df     = df[df["subject_key"].isin(valid)].copy()
    tr, te = [], []
    for _, g in df.groupby("subject_key"):
        idx = g.index.to_numpy().copy(); rng.shuffle(idx)
        n   = max(1, int(len(idx) * frac))
        tr.extend(idx[:n]); te.extend(idx[n:])
    le = LabelEncoder(); le.fit(df["subject_key"].values)
    Xtr = df.loc[tr, feat_cols].values; ytr = le.transform(df.loc[tr, "subject_key"])
    Xte = df.loc[te, feat_cols].values; yte = le.transform(df.loc[te, "subject_key"])
    return Xtr, ytr, Xte, yte, le


def ver_pairs(df, feat_cols, subjs, neg_ratio, rng):
    # Build (features, label) arrays for learned verification.
    protos = {}
    for s, g in df[df["subject_key"].isin(subjs)].groupby("subject_key"):
        idx = g.index.to_numpy().copy(); rng.shuffle(idx)
        h   = max(1, len(idx) // 2)
        a   = np.nanmean(df.loc[idx[:h],    feat_cols].values, axis=0)
        b   = np.nanmean(df.loc[idx[h:2*h], feat_cols].values, axis=0)
        protos[s] = (a, b)
    sl = [s for s in subjs if s in protos]
    if len(sl) < 2:
        return None, None
    pos = [np.abs(protos[s][0] - protos[s][1]) for s in sl]
    neg, att = [], 0
    while len(neg) < len(pos) * neg_ratio and att < len(pos) * neg_ratio * 20:
        s1, s2 = rng.choice(sl, 2, replace=False)
        neg.append(np.abs(protos[s1][0] - protos[s2][1])); att += 1
    if not neg:
        return None, None
    return np.vstack(pos + neg), np.array([1] * len(pos) + [0] * len(neg))


def dist_ver(df, feat_cols, subjs, neg_ratio, rng):
    # Euclidean distance-based verification AUC and EER.
    protos = {}
    for s, g in df[df["subject_key"].isin(subjs)].groupby("subject_key"):
        idx = g.index.to_numpy().copy(); rng.shuffle(idx)
        h   = max(1, len(idx) // 2)
        protos[s] = (np.nanmean(df.loc[idx[:h],    feat_cols].values, axis=0),
                     np.nanmean(df.loc[idx[h:2*h], feat_cols].values, axis=0))
    sl = [s for s in subjs if s in protos]
    scores, labels = [], []
    for s in sl:
        scores.append(-np.linalg.norm(protos[s][0] - protos[s][1])); labels.append(1)
    att = 0
    while len(scores) - len(sl) < len(sl) * neg_ratio and att < len(sl) * neg_ratio * 20:
        s1, s2 = rng.choice(sl, 2, replace=False)
        scores.append(-np.linalg.norm(protos[s1][0] - protos[s2][1])); labels.append(0)
        att += 1
    if len(set(labels)) < 2:
        return np.nan, np.nan
    fpr, tpr, _ = roc_curve(labels, scores)
    return roc_auc_score(labels, scores), eer(fpr, tpr)


print("Helpers ready.")


---
## Section A — Closed-set identification

Segments are split 70/30 **within each subject** independently.  
The same subject appears in both training and test — this characterises
biometric discriminability, not held-out-subject generalisation.


In [ ]:
ident_rows = []

for ds_name, ds in DS.items():
    ft, fc = ds["ft"], ds["feat_cols"]
    n_subj = ft["subject_key"].nunique()
    print(f"\n{ds_name.upper()} ({n_subj} subjects)")

    for rep in range(N_REPEATS):
        Xtr, ytr, Xte, yte, le = seg_split(ft, fc, TRAIN_SEG_FRAC, seed=42 + rep)
        clf   = pipe(ExtraTreesClassifier(**ET)); clf.fit(Xtr, ytr)
        ypred = clf.predict(Xte)
        top1  = accuracy_score(yte, ypred)
        bal   = balanced_accuracy_score(yte, ypred)
        top5  = (top_k_accuracy_score(yte, clf.predict_proba(Xte), k=5,
                                      labels=np.arange(len(le.classes_)))
                 if len(le.classes_) >= 5 else np.nan)
        ident_rows.append({"dataset": ds_name, "repeat": rep,
                           "n_subjects": len(le.classes_),
                           "top1": top1, "balanced": bal, "top5": top5})
        if rep == 0:
            print(f"  rep0  balanced={bal:.4f}  chance={1/len(le.classes_):.4f}")

ident_df = pd.DataFrame(ident_rows)
ident_df.to_csv(OUT_DIR / "A_identification.csv", index=False)

s = ident_df.groupby("dataset")[["top1", "balanced", "top5"]].agg(["mean", "std"]).round(4)
s.columns = ["_".join(c) for c in s.columns]
s["chance"] = (1 / ident_df.groupby("dataset")["n_subjects"].first()).round(4)
s["ratio"]  = (s["balanced_mean"] / s["chance"]).round(1)
print("\n== Summary =="); print(s.to_string())


---
## Section B — Biometric verification (AUC, EER)

Subjects are first split **70/30 at the subject level**.  
Prototypes and pairs are built independently within each group.  
The verifier is fitted on training-group pairs only and evaluated on test-group pairs —
no test subject's data is seen during training.


In [ ]:
ver_rows = []

for ds_name, ds in DS.items():
    ft, fc = ds["ft"], ds["feat_cols"]
    counts = ft.groupby("subject_key").size()
    valid  = counts[counts >= MIN_SEGS].index.tolist()
    ft     = ft[ft["subject_key"].isin(valid)].copy()
    subjs  = np.array(sorted(ft["subject_key"].unique()))
    print(f"\n{ds_name.upper()} ({len(subjs)} subjects)")

    for rep in range(N_REPEATS):
        rng  = np.random.default_rng(42 + rep)
        sh   = subjs.copy(); rng.shuffle(sh)
        n_tr = max(2, int(len(sh) * TRAIN_SUBJ_FRAC))
        tr_s, te_s = sh[:n_tr].tolist(), sh[n_tr:].tolist()

        d_auc, d_eer = dist_ver(ft, fc, te_s, NEG_PAIRS_PER_POS, rng)

        Xtr_p, ytr_p = ver_pairs(ft, fc, tr_s, NEG_PAIRS_PER_POS, rng)
        Xte_p, yte_p = ver_pairs(ft, fc, te_s, NEG_PAIRS_PER_POS, rng)
        l_auc = l_eer = np.nan
        if Xtr_p is not None and Xte_p is not None and len(set(yte_p)) == 2:
            clf   = pipe(ExtraTreesClassifier(**ET)); clf.fit(Xtr_p, ytr_p)
            pr    = clf.predict_proba(Xte_p)[:, 1]
            l_auc = roc_auc_score(yte_p, pr)
            fpr, tpr, _ = roc_curve(yte_p, pr); l_eer = eer(fpr, tpr)

        ver_rows.append({"dataset": ds_name, "repeat": rep,
                         "n_tr_subj": len(tr_s), "n_te_subj": len(te_s),
                         "dist_auc": d_auc, "dist_eer": d_eer,
                         "learned_auc": l_auc, "learned_eer": l_eer})
        if rep == 0:
            print(f"  rep0  dist_auc={d_auc:.4f}  learned_auc={l_auc:.4f}")

ver_df = pd.DataFrame(ver_rows)
ver_df.to_csv(OUT_DIR / "B_verification.csv", index=False)
print("\n== Summary ==")
print(ver_df.groupby("dataset")[["dist_auc", "dist_eer", "learned_auc", "learned_eer"]]
      .agg(["mean", "std"]).round(4).to_string())


---
## Section C — Distance decomposition

Computes within-subject (cross-session/condition) vs between-subject (same-session/condition)
Euclidean distances in the standardised feature space.  
Tests whether within-subject distances are smaller using a **two-sided** Mann-Whitney *U* test.


In [ ]:
dist_rows = []

for ds_name, ds in DS.items():
    ft, fc = ds["ft"].copy(), ds["feat_cols"]

    imp = SimpleImputer(strategy="median"); sc = StandardScaler()
    Xsc = sc.fit_transform(imp.fit_transform(ft[fc].values))
    ft["_row"] = np.arange(len(ft))

    cond_col = None
    for col in ["session", "condition", "run", "run_type"]:
        if col in ft.columns and ft[col].nunique() > 1:
            cond_col = col; break
    if cond_col is None:
        ft["_half"] = (np.arange(len(ft)) % 2).astype(str); cond_col = "_half"

    protos = {}
    for (subj, cond), g in ft.groupby(["subject_key", cond_col]):
        protos[(subj, cond)] = Xsc[g["_row"].values].mean(axis=0)

    subjects   = sorted(ft["subject_key"].unique())
    conditions = sorted(ft[cond_col].unique())
    within_d, between_d = [], []
    for s1 in subjects:
        for s2 in subjects:
            for c1 in conditions:
                for c2 in conditions:
                    if (s1, c1) not in protos or (s2, c2) not in protos: continue
                    if s1 == s2 and c1 == c2: continue
                    d = float(np.linalg.norm(protos[(s1, c1)] - protos[(s2, c2)]))
                    if s1 == s2:    within_d.append(d)
                    elif c1 == c2:  between_d.append(d)

    if not within_d or not between_d:
        print(f"  {ds_name}: insufficient pairs"); continue

    wm, ws = np.mean(within_d), np.std(within_d)
    bm, bs = np.mean(between_d), np.std(between_d)
    ratio  = wm / bm
    _, pval = mannwhitneyu(within_d, between_d, alternative="two-sided")
    holds  = wm < bm

    dist_rows.append({"dataset": ds_name, "cond_col": cond_col,
                      "within_mean": round(wm, 4), "within_std": round(ws, 4),
                      "between_mean": round(bm, 4), "between_std": round(bs, 4),
                      "ratio": round(ratio, 4), "pval_twosided": round(pval, 6),
                      "holds": holds, "n_within": len(within_d), "n_between": len(between_d)})
    print(f"  {ds_name:<15}  ratio={ratio:.3f}  p={pval:.6f} (two-sided)  "
          f"{'within < between ✓' if holds else 'within >= between ✗'}")

dist_df = pd.DataFrame(dist_rows)
dist_df.to_csv(OUT_DIR / "C_distance_decomposition.csv", index=False)
print("\nSaved C_distance_decomposition.csv")


---
## Section D — Cross-session identity stability (Longitudinal ERP)

Trains on one session, tests on another, across all six pairwise session combinations.  
Group A only (15 subjects with data across all four sessions).


In [ ]:
SESSION_ORDER = ["day1", "day7", "day80", "day200"]
DAY_MAP       = {"day1": 1, "day7": 7, "day80": 80, "day200": 200}

ft_l = DS["longitudinal"]["ft"].copy()
fc_l = DS["longitudinal"]["feat_cols"]
ft_a = ft_l[ft_l["group"] == "A"].copy() if "group" in ft_l.columns else ft_l.copy()

xses_rows = []
pairs = [(SESSION_ORDER[i], SESSION_ORDER[j]) for i in range(4) for j in range(i + 1, 4)]

print("Cross-session identity (Group A, 15 subjects)")
print(f"  {'Train→Test':<22} {'Bal.Acc':>8}  {'Chance':>8}  {'Gap':>6}")
print("  " + "-" * 50)

for tr_s, te_s in pairs:
    tr = ft_a[ft_a["session"] == tr_s]
    te = ft_a[ft_a["session"] == te_s]
    common = sorted(set(tr["subject_key"]) & set(te["subject_key"]))
    if len(common) < 2: continue
    tr = tr[tr["subject_key"].isin(common)]
    te = te[te["subject_key"].isin(common)]
    le  = LabelEncoder(); le.fit(common)
    gap = DAY_MAP[te_s] - DAY_MAP[tr_s]
    accs = []
    for rep in range(5):
        clf = pipe(ExtraTreesClassifier(**ET))
        clf.fit(tr[fc_l].values, le.transform(tr["subject_key"].values))
        accs.append(balanced_accuracy_score(le.transform(te["subject_key"].values),
                                            clf.predict(te[fc_l].values)))
        xses_rows.append({"train": tr_s, "test": te_s, "days": gap, "repeat": rep,
                          "n_subj": len(common), "bal": accs[-1], "chance": 1 / len(common)})
    print(f"  {tr_s}→{te_s:<15}  {np.mean(accs):.4f}   {1/len(common):.4f}   {gap:>4}d")

# Within-session ceiling
df_d1 = ft_a[ft_a["session"] == "day1"].copy()
rng   = np.random.default_rng(42)
tri, tei = [], []
for _, g in df_d1.groupby("subject_key"):
    idx = g.index.to_numpy().copy(); rng.shuffle(idx)
    n   = max(1, int(len(idx) * 0.7)); tri.extend(idx[:n]); tei.extend(idx[n:])
le_ws  = LabelEncoder(); le_ws.fit(df_d1["subject_key"].values)
clf_ws = pipe(ExtraTreesClassifier(**ET))
clf_ws.fit(df_d1.loc[tri, fc_l].values, le_ws.transform(df_d1.loc[tri, "subject_key"].values))
bal_ws = balanced_accuracy_score(le_ws.transform(df_d1.loc[tei, "subject_key"].values),
                                 clf_ws.predict(df_d1.loc[tei, fc_l].values))
print(f"\n  Within-session ceiling (day1 seg split): {bal_ws:.4f}  (chance=0.067)")

xses_df = pd.DataFrame(xses_rows)
xses_df.to_csv(OUT_DIR / "D_cross_session.csv", index=False)
print("Saved D_cross_session.csv")


---
## Section E — Cross-paradigm identity transfer

Tests whether identity trained in one recording condition transfers to another.  
All subjects appear in both train and test; only the condition is held out.


In [ ]:
transfer_rows = []

# ── PhysioNet: resting baseline → motor task ──────────────────────────────────
ft_pn, fc_pn = DS["physionet"]["ft"].copy(), DS["physionet"]["feat_cols"]
df_base = ft_pn[ft_pn["run_type"] == "baseline"]
df_task = ft_pn[ft_pn["run_type"] == "task"]
le_pn   = LabelEncoder(); le_pn.fit(ft_pn["subject_key"].values)
print("PhysioNet: baseline → task")

for rep in range(N_REPEATS):
    clf = pipe(ExtraTreesClassifier(n_estimators=300, random_state=42 + rep, n_jobs=-1))
    clf.fit(df_base[fc_pn].values, le_pn.transform(df_base["subject_key"].values))
    bal = balanced_accuracy_score(le_pn.transform(df_task["subject_key"].values),
                                  clf.predict(df_task[fc_pn].values))
    transfer_rows.append({"dataset": "physionet", "train": "baseline", "test": "task",
                          "repeat": rep, "n_subj": len(le_pn.classes_),
                          "bal": bal, "chance": 1 / len(le_pn.classes_)})

pn_bal = np.mean([r["bal"] for r in transfer_rows if r["dataset"] == "physionet"])
pn_ch  = 1 / len(le_pn.classes_)
print(f"  Mean balanced acc: {pn_bal:.4f}  chance={pn_ch:.4f}  ratio={pn_bal/pn_ch:.1f}×")

# ── SignEEG: genuine signing → forged signing ─────────────────────────────────
ft_sg, fc_sg = DS["signeeg"]["ft"].copy(), DS["signeeg"]["feat_cols"]
print("\nSignEEG: genuine → forged  (note: identifiers provisional — interpret cautiously)")

if "genuine_signature" in ft_sg["condition"].values and "forged_signature" in ft_sg["condition"].values:
    df_gen  = ft_sg[ft_sg["condition"] == "genuine_signature"]
    df_forg = ft_sg[ft_sg["condition"] == "forged_signature"]
    le_sg   = LabelEncoder(); le_sg.fit(ft_sg["subject_key"].values)
    for rep in range(N_REPEATS):
        clf = pipe(ExtraTreesClassifier(n_estimators=300, random_state=42 + rep, n_jobs=-1))
        clf.fit(df_gen[fc_sg].values, le_sg.transform(df_gen["subject_key"].values))
        bal = balanced_accuracy_score(le_sg.transform(df_forg["subject_key"].values),
                                      clf.predict(df_forg[fc_sg].values))
        transfer_rows.append({"dataset": "signeeg", "train": "genuine", "test": "forged",
                               "repeat": rep, "n_subj": len(le_sg.classes_),
                               "bal": bal, "chance": 1 / len(le_sg.classes_)})
    sg_bal = np.mean([r["bal"] for r in transfer_rows if r["dataset"] == "signeeg"])
    sg_ch  = 1 / len(le_sg.classes_)
    print(f"  Mean balanced acc: {sg_bal:.4f}  chance={sg_ch:.4f}  ratio={sg_bal/sg_ch:.2f}×")

transfer_df = pd.DataFrame(transfer_rows)
transfer_df.to_csv(OUT_DIR / "E_cross_paradigm_transfer.csv", index=False)
print("\nSaved E_cross_paradigm_transfer.csv")


---
## Section F — Leakage simulation (Mumtaz MDD)

Compares MDD vs healthy classification under two evaluation protocols:

- **Naive:** segments split 70/30 randomly (same subject can appear in both train and test)
- **Correct:** subjects split 70/30 (no subject overlap across train and test)

Also reports:
- **Subject-level AUC** (segment probabilities averaged per subject before ROC)
- **Permutation test** (labels shuffled at subject level, N=200)
- **Identity-derived AUC** (subject identity classifier → clinical label mapping)

All *p*-values are two-sided. The StandardScaler is fitted inside the pipeline on training data only.


In [ ]:
ft_mz = DS["mumtaz"]["ft"].copy().reset_index(drop=True)
fc_mz = DS["mumtaz"]["feat_cols"]
ft_mz["mdd_label"] = (ft_mz["label"].str.upper() == "MDD").astype(int)

N_PERM   = 200
LR_MAIN  = dict(max_iter=1000, random_state=42, C=1.0, solver="lbfgs", n_jobs=-1)
LR_PERM  = dict(max_iter=300,  random_state=42, C=1.0, solver="lbfgs", n_jobs=-1)


def run_mdd(df, protocol, rng, lr_kwargs=None, permute_labels=False):
    """Return (seg_auc, subj_auc) for one repeat, or None if test set is not binary."""
    if lr_kwargs is None: lr_kwargs = LR_MAIN
    if protocol == "segment":
        idx  = np.arange(len(df)); rng.shuffle(idx)
        n_tr = int(len(idx) * 0.70)
        tr, te = df.iloc[idx[:n_tr]], df.iloc[idx[n_tr:]]
    else:
        subjs = np.array(sorted(df["subject_key"].unique())); rng.shuffle(subjs)
        n_tr  = max(2, int(len(subjs) * 0.70))
        tr = df[df["subject_key"].isin(subjs[:n_tr])]
        te = df[df["subject_key"].isin(subjs[n_tr:])]
    if len(set(te["mdd_label"])) < 2: return None

    y_tr = tr["mdd_label"].values.copy()
    if permute_labels:
        su   = sorted(tr["subject_key"].unique())
        orig = {s: tr[tr["subject_key"] == s]["mdd_label"].iloc[0] for s in su}
        pmap = dict(zip(su, [orig[k] for k in rng.permutation(su)]))
        y_tr = np.array([pmap[s] for s in tr["subject_key"].values])

    clf      = Pipeline([("imp", SimpleImputer(strategy="median")),
                         ("sc",  StandardScaler()),
                         ("lr",  LogisticRegression(**lr_kwargs))])
    clf.fit(tr[fc_mz].values, y_tr)
    prob_seg = clf.predict_proba(te[fc_mz].values)[:, 1]
    seg_auc  = roc_auc_score(te["mdd_label"].values, prob_seg)
    te2 = te.assign(prob=prob_seg)
    sp  = te2.groupby("subject_key")["prob"].mean()
    sl  = te2.groupby("subject_key")["mdd_label"].first()
    subj_auc = roc_auc_score(sl.values, sp.values) if len(set(sl.values)) == 2 else np.nan
    return seg_auc, subj_auc


# ── Main 10-repeat experiment ─────────────────────────────────────────────────
print("Running 10-repeat MDD experiment …")
mdd_rows = []
for protocol in ["segment", "subject"]:
    for rep in range(N_REPEATS):
        res = run_mdd(ft_mz, protocol, np.random.default_rng(42 + rep))
        if res:
            mdd_rows.append({"protocol": protocol, "rep": rep,
                             "seg_auc": res[0], "subj_auc": res[1]})
    print(f"  {protocol}: done")

df_mdd    = pd.DataFrame(mdd_rows)
seg_rows  = df_mdd[df_mdd["protocol"] == "segment"]
subj_rows = df_mdd[df_mdd["protocol"] == "subject"]

# ── Permutation test (subject-wise, N=200) ────────────────────────────────────
print(f"\nRunning {N_PERM} subject-level permutations …")
perm_aucs = []
for p in range(N_PERM):
    res = run_mdd(ft_mz, "subject", np.random.default_rng(1000 + p),
                  lr_kwargs=LR_PERM, permute_labels=True)
    if res: perm_aucs.append(res[1])
    if (p + 1) % 50 == 0: print(f"  {p+1}/{N_PERM}")

perm_arr = np.array(perm_aucs)
obs_auc  = subj_rows["subj_auc"].mean()
perm_p   = np.mean(perm_arr >= obs_auc)

# ── Identity-derived AUC diagnostic ──────────────────────────────────────────
rng    = np.random.default_rng(42)
idx    = np.arange(len(ft_mz)); rng.shuffle(idx)
n_tr   = int(len(idx) * 0.70)
tr_id  = ft_mz.iloc[idx[:n_tr]]
te_id  = ft_mz.iloc[idx[n_tr:]]
le_id  = LabelEncoder(); le_id.fit(ft_mz["subject_key"].values)
id_clf = Pipeline([("imp", SimpleImputer(strategy="median")),
                   ("sc",  StandardScaler()),
                   ("et",  ExtraTreesClassifier(**ET))])
id_clf.fit(tr_id[fc_mz].values, le_id.transform(tr_id["subject_key"].values))
subj_mdd  = ft_mz.drop_duplicates("subject_key").set_index("subject_key")["mdd_label"].to_dict()
mdd_vec   = np.array([subj_mdd.get(s, 0) for s in le_id.classes_], dtype=float)
mdd_score = id_clf.predict_proba(te_id[fc_mz].values) @ mdd_vec
id_auc    = roc_auc_score(te_id["mdd_label"].values, mdd_score)

# ── Results ───────────────────────────────────────────────────────────────────
_, p2s_seg  = mannwhitneyu(seg_rows["seg_auc"],  subj_rows["seg_auc"],  alternative="two-sided")
_, p2s_subj = mannwhitneyu(seg_rows["subj_auc"].dropna(),
                            subj_rows["subj_auc"].dropna(), alternative="two-sided")

print("\n=== Segment-level AUC ===")
print(f"  Naive   (segment split): {seg_rows['seg_auc'].mean():.4f} ± {seg_rows['seg_auc'].std():.4f}")
print(f"  Correct (subject split): {subj_rows['seg_auc'].mean():.4f} ± {subj_rows['seg_auc'].std():.4f}")
print(f"  Inflation: {seg_rows['seg_auc'].mean()-subj_rows['seg_auc'].mean():+.4f}  p={p2s_seg:.4f} (two-sided)")
print("\n=== Subject-level AUC (primary unit) ===")
print(f"  Naive   (segment split): {seg_rows['subj_auc'].mean():.4f} ± {seg_rows['subj_auc'].std():.4f}")
print(f"  Correct (subject split): {subj_rows['subj_auc'].mean():.4f} ± {subj_rows['subj_auc'].std():.4f}")
print(f"  Inflation: {seg_rows['subj_auc'].mean()-subj_rows['subj_auc'].mean():+.4f}  p={p2s_subj:.4f} (two-sided)")
print(f"\n=== Permutation test (N={len(perm_arr)}) ===")
print(f"  Observed subj AUC: {obs_auc:.4f}")
print(f"  Null mean:         {perm_arr.mean():.4f} ± {perm_arr.std():.4f}")
print(f"  p-value:           {perm_p:.4f}  (fraction of null ≥ observed)")
print(f"\n=== Identity-derived AUC diagnostic ===")
print(f"  MDD AUC via identity classifier (no clinical labels): {id_auc:.4f}")

df_mdd.to_csv(OUT_DIR / "F_mdd_leakage.csv", index=False)
pd.DataFrame({"perm_subj_auc": perm_arr}).to_csv(OUT_DIR / "F_mdd_permutations.csv", index=False)
json.dump({"naive_seg_auc": round(float(seg_rows["seg_auc"].mean()), 4),
           "naive_subj_auc": round(float(seg_rows["subj_auc"].mean()), 4),
           "correct_seg_auc": round(float(subj_rows["seg_auc"].mean()), 4),
           "correct_subj_auc": round(float(subj_rows["subj_auc"].mean()), 4),
           "inflation_seg": round(float(seg_rows["seg_auc"].mean()-subj_rows["seg_auc"].mean()), 4),
           "inflation_subj": round(float(seg_rows["subj_auc"].mean()-subj_rows["subj_auc"].mean()), 4),
           "p_twosided_seg": round(float(p2s_seg), 4),
           "p_twosided_subj": round(float(p2s_subj), 4),
           "permutation_p": round(float(perm_p), 4),
           "n_permutations": len(perm_arr),
           "identity_derived_auc": round(float(id_auc), 4)},
          open(OUT_DIR / "F_mdd_leakage.json", "w"), indent=2)
print("\nSaved F_mdd_leakage.csv / .json / permutations.csv")


---
## Section G — Summary tables

Prints and saves all six paper tables as CSV.

In [ ]:
sep = "=" * 68

print(sep); print("TABLE 1 — Closed-set identification"); print(sep)
t1 = ident_df.groupby("dataset").agg(
    N=("n_subjects", "first"),
    bal_mean=("balanced", "mean"), bal_std=("balanced", "std"),
    top5_mean=("top5", "mean"),
).round(4)
t1["chance"]  = (1 / t1["N"]).round(4)
t1["ratio_x"] = (t1["bal_mean"] / t1["chance"]).round(1)
print(t1.to_string()); t1.to_csv(OUT_DIR / "TABLE1_identification.csv")

print(); print(sep); print("TABLE 2 — Verification"); print(sep)
t2 = ver_df.groupby("dataset").agg(
    dist_auc_mean=("dist_auc", "mean"), dist_auc_std=("dist_auc", "std"),
    dist_eer_mean=("dist_eer", "mean"),
    learned_auc_mean=("learned_auc", "mean"), learned_auc_std=("learned_auc", "std"),
    learned_eer_mean=("learned_eer", "mean"),
).round(4)
print(t2.to_string()); t2.to_csv(OUT_DIR / "TABLE2_verification.csv")

print(); print(sep); print("TABLE 3 — Distance decomposition (two-sided MW)"); print(sep)
print(dist_df[["dataset", "within_mean", "within_std",
               "between_mean", "between_std", "ratio", "pval_twosided", "holds"]].to_string(index=False))
dist_df.to_csv(OUT_DIR / "TABLE3_distance.csv", index=False)

print(); print(sep); print("TABLE 4 — Cross-session stability"); print(sep)
t4 = xses_df.groupby(["train", "test", "days", "chance"])["bal"].agg(
    mean="mean", std="std").round(4).reset_index()
print(t4.to_string(index=False)); t4.to_csv(OUT_DIR / "TABLE4_cross_session.csv", index=False)

print(); print(sep); print("TABLE 5 — Cross-paradigm transfer"); print(sep)
t5 = transfer_df.groupby(["dataset", "train", "test", "n_subj", "chance"])["bal"].agg(
    mean="mean", std="std").round(4).reset_index()
t5["ratio_x"] = (t5["mean"] / t5["chance"]).round(1)
print(t5.to_string(index=False)); t5.to_csv(OUT_DIR / "TABLE5_transfer.csv", index=False)

print(); print(sep); print("TABLE 6 — MDD leakage"); print(sep)
t6 = df_mdd.groupby("protocol").agg(
    seg_auc_mean=("seg_auc", "mean"), seg_auc_std=("seg_auc", "std"),
    subj_auc_mean=("subj_auc", "mean"), subj_auc_std=("subj_auc", "std"),
).round(4)
print(t6.to_string()); t6.to_csv(OUT_DIR / "TABLE6_mdd_leakage.csv")

print(f"\nAll CSVs saved to: {OUT_DIR}")


---
## Section I — Variance decomposition

Two-way method-of-moments variance decomposition applied to each spectral feature:

$$x_{ijk} = \mu + \alpha_i + \gamma_j + \varepsilon_{ijk}$$

where $\alpha_i$ = subject identity, $\gamma_j$ = session/run effect.

Confidence intervals use **subject-level bootstrapping** (resample subjects with replacement),
which correctly propagates subject-level uncertainty.

Run on two datasets:
1. **Longitudinal ERP** — 15 subjects × 4 sessions (Group A)
2. **PhysioNet EEGMMIDB** — 103 subjects × 14 runs (replication)


In [ ]:
def icc_all_features(M):

    n_s, n_j, n_f = M.shape
    complete = ~np.isnan(M).any(axis=(1, 2))
    m  = M[complete]; nv = m.shape[0]
    if nv < 3:
        nan = np.full(n_f, np.nan); return nan, nan, nan, nan

    grand = m.mean(axis=(0, 1))
    sm    = m.mean(axis=1)
    jm    = m.mean(axis=0)
    SS_s  = n_j * np.sum((sm - grand) ** 2, axis=0)
    SS_j  = nv  * np.sum((jm - grand) ** 2, axis=0)
    SS_e  = np.sum((m - grand) ** 2, axis=(0, 1)) - SS_s - SS_j
    MS_s  = SS_s / (nv - 1)
    MS_j  = SS_j / (n_j - 1)
    MS_e  = SS_e / ((nv - 1) * (n_j - 1))
    var_e = np.maximum(MS_e, 0.0)
    var_j = np.maximum((MS_j - var_e) / nv,  0.0)
    var_s = np.maximum((MS_s - var_e) / n_j, 0.0)
    tot   = var_s + var_j + var_e + 1e-30
    return var_s / tot, var_s / tot, var_j / tot, var_e / tot


def variance_decomposition(ft, fc, subjects, session_col, sessions, label="Dataset"):
    print(f"\n=== {label} ===")
    ft = ft.reset_index(drop=True)

    # Cell balance
    cb = ft.groupby(["subject_key", session_col]).size().unstack(fill_value=0)
    print(f"  Subjects  : {len(subjects)}")
    print(f"  Sessions  : {len(sessions)}")
    print(f"  Features  : {len(fc)}")
    print(f"  Epochs per cell — min={cb.values.min()}  "
          f"mean={cb.values.mean():.1f}  max={cb.values.max()}  "
          f"empty={int((cb.values == 0).sum())}")

    # Impute, build cell-mean array
    X  = SimpleImputer(strategy="median").fit_transform(ft[fc].values)
    si = {s: i for i, s in enumerate(subjects)}
    ji = {s: j for j, s in enumerate(sessions)}
    M_sum = np.zeros((len(subjects), len(sessions), len(fc)))
    M_cnt = np.zeros((len(subjects), len(sessions)), dtype=int)
    for row in range(len(ft)):
        s = ft["subject_key"].iloc[row]; ses = ft[session_col].iloc[row]
        if s in si and ses in ji:
            M_sum[si[s], ji[ses]] += X[row]; M_cnt[si[s], ji[ses]] += 1
    M_all = np.where(M_cnt[:, :, None] > 0, M_sum / M_cnt[:, :, None], np.nan)

    icc_v, p_s_v, p_j_v, p_e_v = icc_all_features(M_all)
    valid = ~np.isnan(icc_v)
    icc = icc_v[valid]; p_s = p_s_v[valid]; p_j = p_j_v[valid]; p_e = p_e_v[valid]
    print(f"  Features with valid ICC: {valid.sum()} / {len(fc)}")

    print(f"  Subject identity : {p_s.mean()*100:.1f}%  "
          f"[{np.percentile(p_s,5)*100:.1f}–{np.percentile(p_s,95)*100:.1f}%]")
    print(f"  Session drift    : {p_j.mean()*100:.1f}%  "
          f"[{np.percentile(p_j,5)*100:.1f}–{np.percentile(p_j,95)*100:.1f}%]")
    print(f"  Residual         : {p_e.mean()*100:.1f}%  "
          f"[{np.percentile(p_e,5)*100:.1f}–{np.percentile(p_e,95)*100:.1f}%]")

    # Subject-level bootstrap
    rng = np.random.default_rng(42)
    boot = np.empty(2000)
    for b in range(2000):
        idx  = rng.integers(0, len(subjects), size=len(subjects))
        M_b  = M_all[idx, :, :]
        ib, _, _, _ = icc_all_features(M_b)
        boot[b] = np.nanmean(ib)
    ci_lo, ci_hi = np.percentile(boot, 2.5), np.percentile(boot, 97.5)
    print(f"  ICC(2,1) mean    : {icc.mean():.4f}")
    print(f"  ICC(2,1) median  : {np.median(icc):.4f}")
    print(f"  95% CI (subj)    : [{ci_lo:.4f}, {ci_hi:.4f}]")

    return {
        "n_subjects": len(subjects), "n_sessions": len(sessions),
        "n_features": int(valid.sum()),
        "cell_min": int(cb.values.min()), "cell_max": int(cb.values.max()),
        "prop_subject": round(float(p_s.mean()), 4),
        "prop_session": round(float(p_j.mean()), 4),
        "prop_residual": round(float(p_e.mean()), 4),
        "icc_mean": round(float(icc.mean()), 4),
        "icc_median": round(float(np.median(icc)), 4),
        "ci_lo": round(ci_lo, 4), "ci_hi": round(ci_hi, 4),
    }, pd.DataFrame({"icc": icc, "prop_subject": p_s, "prop_session": p_j, "prop_residual": p_e})


# ── Longitudinal ERP (Group A) ────────────────────────────────────────────────
ft_l  = DS["longitudinal"]["ft"].copy()
ft_a  = ft_l[ft_l["group"] == "A"].copy() if "group" in ft_l.columns else ft_l.copy()
ft_a  = ft_a[ft_a["session"].isin(["day1","day7","day80","day200"])].copy()
res_l, feat_l = variance_decomposition(
    ft_a, DS["longitudinal"]["feat_cols"],
    sorted(ft_a["subject_key"].unique()), "session",
    ["day1","day7","day80","day200"], label="Longitudinal ERP — Group A")

json.dump(res_l, open(OUT_DIR / "I_variance_longitudinal.json", "w"), indent=2)
feat_l.to_csv(OUT_DIR / "I_variance_longitudinal.csv", index=False)

# ── PhysioNet EEGMMIDB (replication) ─────────────────────────────────────────
ft_pn = DS["physionet"]["ft"].copy()
run_col = next((c for c in ["run","run_id","session","condition"]
                if c in ft_pn.columns and ft_pn[c].nunique() >= 4), None)
if run_col is None:
    print("Could not find run column in PhysioNet dataset.")
else:
    runs_pn = sorted(ft_pn[run_col].unique())
    res_pn, feat_pn = variance_decomposition(
        ft_pn, DS["physionet"]["feat_cols"],
        sorted(ft_pn["subject_key"].unique()), run_col,
        runs_pn, label="PhysioNet EEGMMIDB — replication")
    json.dump(res_pn, open(OUT_DIR / "I_variance_physionet.json", "w"), indent=2)
    feat_pn.to_csv(OUT_DIR / "I_variance_physionet.csv", index=False)

print("\nSaved I_variance_longitudinal.json/.csv and I_variance_physionet.json/.csv")


---
## Results summary

All output files are saved to `OUT_DIR` (configured in Section 0).

In [ ]:
print(f"Output directory: {OUT_DIR}\n")
for p in sorted(OUT_DIR.iterdir()):
    size_kb = p.stat().st_size / 1024
    print(f"  {p.name:<45} {size_kb:>7.1f} KB")
